In [ ]:
# Imports
import json
import concurrent.futures
import re
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
from anthropic import Anthropic
from collections import defaultdict

In [ ]:
# Client Initialization and helper functions

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# Report Builder
def generate_prompt_evaluation_report(evaluation_results):
    import os
    import html
    from statistics import mean
    
    total_tests = len(evaluation_results)
    scores = [result.get("total_score", 0) for result in evaluation_results]
    avg_score = mean(scores) if scores else 0
    max_possible_score = 40
    pass_rate = (
        100 * len([s for s in scores if s >= 30]) / total_tests if total_tests else 0
    )

    template_path = "report_template.html"
    if not os.path.exists(template_path):
        return "Error: report_template.html not found. Please ensure it exists in the same directory."
        
    with open(template_path, "r", encoding="utf-8") as f:
        html_template = f.read()
        
    table_rows = ""
    for result in evaluation_results:
        prompt_inputs_html = "<br>".join(
            [
                f"<strong>{key}:</strong> {value}"
                for key, value in result["test_case"]["prompt_inputs"].items()
            ]
        )

        criteria_string = "<br>• ".join(result["test_case"]["solution_criteria"])

        total_score = result.get("total_score", 0)
        metrics = result.get("scores", {})
        
        if total_score >= 32:
            score_class = "score-high"
        elif total_score <= 20:
            score_class = "score-low"
        else:
            score_class = "score-medium"

        escaped_output = html.escape(result["output"])
        
        metric_html = ""
        for key, val in metrics.items():
            short_key = key.replace("_", " ").title()
            if short_key == "Api Integration": short_key = "API"
            elif short_key == "Prompt Adherence": short_key = "Prompt"
            elif short_key == "Architecture": short_key = "Arch"
            elif short_key == "Completeness": short_key = "Code"
            
            metric_html += f'''
                <div class="metric">
                    <span class="metric-label">{short_key}</span>
                    <span class="metric-value">{val}</span>
                </div>
            '''

        table_rows += f"""
            <div class="grid-row">
                <div class="grid-cell">{result["test_case"]["scenario"]}</div>
                <div class="grid-cell prompt-inputs">{prompt_inputs_html}</div>
                <div class="grid-cell criteria">• {criteria_string}</div>
                <div class="grid-cell output">
                    <div class="output-container">
                        <button type="button" class="copy-btn" title="Copy output">
                            💾 COPY
                        </button>
                        <pre>{escaped_output}</pre>
                    </div>
                </div>
                <div class="grid-cell score-col">
                    <span class="score {score_class}">{total_score}</span>
                    <div class="score-breakdown">
                        {metric_html}
                    </div>
                </div>
                <div class="grid-cell reasoning">{html.escape(result["reasoning"])}</div>
            </div>
        """

    html_template = html_template.replace("__TOTAL_TESTS__", str(total_tests))
    html_template = html_template.replace("__AVG_SCORE__", f"{avg_score:.1f}")
    html_template = html_template.replace("__PASS_RATE__", f"{pass_rate:.1f}")
    html_template = html_template.replace("__TABLE_ROWS__", table_rows)

    return html_template


In [ ]:
class SafeDict(dict):
    def __missing__(self, key):
        return "{" + key + "}"

In [ ]:
# PromptEvaluator Implementation
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks

    def render(self, template_string, variables):
        return template_string.format_map(SafeDict(variables))

    def generate_unique_ideas(self, task_description, prompt_inputs_spec, num_cases):
        """Generate a list of unique ideas for test cases based on the task description"""

        prompt = """
        Generate {num_cases} unique, diverse ideas for testing a prompt that accomplishes this task:
        
        <task_description>
        {task_description}
        </task_description>

        The prompt will receive the following inputs
        <prompt_inputs>
        {prompt_inputs_spec}
        </prompt_inputs>
        
        Each idea should represent a distinct scenario or example that tests different aspects of the task.
        
        Output Format:
        Provide your response as a structured JSON array where each item is a brief description of the idea.
        
        Example:
        ```json
        [
            "Testing with technical computer science terminology",
            "Testing with medical research findings",
            "Testing with complex mathematical concepts",
            ...
        ]
        ```
        
        Ensure each idea is:
        - Clearly distinct from the others
        - Relevant to the task description
        - Specific enough to guide generation of a full test case
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output

        Remember, only generate {num_cases} unique ideas
        """

        system_prompt = "You are a test scenario designer specialized in creating diverse, unique testing scenarios."

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": str # {val},'

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "task_description": task_description,
                "num_cases": num_cases,
                "prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
            temperature=1.0,
        )

        return json.loads(text)

    def generate_test_case(self, task_description, idea, prompt_inputs_spec={}):
        """Generate a single test case based on the task description and a specific idea"""

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": "EXAMPLE_VALUE", // {val}\n'

        allowed_keys = ", ".join([f'"{key}"' for key in prompt_inputs_spec.keys()])

        prompt = """
        Generate a single detailed test case for a prompt evaluation based on:
        
        <task_description>
        {task_description}
        </task_description>
        
        <specific_idea>
        {idea}
        </specific_idea>
        
        <allowed_input_keys>
        {allowed_keys}
        </allowed_input_keys>
        
        Output Format:
        ```json
        {{
            "prompt_inputs": {{
            {example_prompt_inputs}
            }},
            "solution_criteria": ["criterion 1", "criterion 2", ...] // Concise list of criteria for evaluating the solution, 1 to 4 items
        }}
        ```

        IMPORTANT REQUIREMENTS:
        - You MUST ONLY use these exact input keys in your prompt_inputs: {allowed_keys}        
        - Do NOT add any additional keys to prompt_inputs
        - All keys listed in allowed_input_keys must be included in your response
        - Make the test case realistic and practically useful
        - Include measurable, concise solution criteria
        - The solution criteria should ONLY address the direct requirements of the task description and the generated prompt_inputs
        - Avoid over-specifying criteria with requirements that go beyond the core task
        - Keep solution criteria simple, focused, and directly tied to the fundamental task
        - The test case should be tailored to the specific idea provided
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output
        - DO NOT include any fields beyond those specified in the output format

        Here's an example of a sample input with an ideal output:
        <sample_input>
        <sample_task_description>
        Extract topics out of a passage of text
        </sample_task_description>
        <sample_specific_idea>
        Testing with a text that contains multiple nested topics and subtopics (e.g., a passage about renewable energy that covers solar power economics, wind turbine technology, and policy implications simultaneously)
        </sample_specific_idea>

        <sample_allowed_input_keys>
        "content"
        </sample_allowed_input_keys>
        </sample_input>
        <ideal_output>
        ```json
        {{
            "prompt_inputs": {{
                "content": "The transition to renewable energy encompasses numerous interdependent dimensions. Solar photovoltaic technology has seen dramatic cost reductions, with panel efficiency improving 24% since 2010 while manufacturing costs declined by 89%, making it economically competitive with fossil fuels in many markets. Concurrently, wind energy has evolved through innovative turbine designs featuring carbon-fiber composite blades and advanced control systems that increase energy capture by 35% in low-wind conditions."
            }},
            "solution_criteria": [
                "Includes all topics mentioned"   
            ]
        }}
        ```
        </ideal_output>
        This is ideal output because the solution criteria is concise and doesn't ask for anything outside of the scope of the task description.
        """

        system_prompt = "You are a test case creator specializing in designing evaluation scenarios."

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "allowed_keys": allowed_keys,
                "task_description": task_description,
                "idea": idea,
                "example_prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
            temperature=0.7,
        )

        test_case = json.loads(text)
        test_case["task_description"] = task_description
        test_case["scenario"] = idea

        return test_case

    def generate_dataset(
        self,
        task_description,
        prompt_inputs_spec={},
        num_cases=1,
        output_file="dataset.json",
    ):
        """Generate test dataset based on task description and save to file"""
        ideas = self.generate_unique_ideas(
            task_description, prompt_inputs_spec, num_cases
        )

        dataset = []
        completed = 0
        total = len(ideas)
        last_reported_percentage = 0

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_idea = {
                executor.submit(
                    self.generate_test_case,
                    task_description,
                    idea,
                    prompt_inputs_spec,
                ): idea
                for idea in ideas
            }

            for future in concurrent.futures.as_completed(future_to_idea):
                try:
                    result = future.result()
                    completed += 1
                    current_percentage = int((completed / total) * 100)
                    milestone_percentage = (current_percentage // 20) * 20

                    if milestone_percentage > last_reported_percentage:
                        print(f"Generated {completed}/{total} test cases")
                        last_reported_percentage = milestone_percentage

                    dataset.append(result)
                except Exception as e:
                    print(f"Error generating test case: {e}")

        with open(output_file, "w") as f:
            json.dump(dataset, f, indent=2)

        return dataset

    def grade_output(self, test_case, output, extra_criteria):
        """Grade the output of a test case using the model"""

        prompt_inputs = ""
        for key, value in test_case["prompt_inputs"].items():
            val = value.replace("\n", "\\n")
            prompt_inputs += f'"{key}":"{val}",\n'

        extra_criteria_section = ""
        if extra_criteria:
            extra_criteria_template = """
            Mandatory Requirements - ANY VIOLATION MEANS AUTOMATIC FAILURE (score of 3 or lower):
            <extra_important_criteria>
            {extra_criteria}
            </extra_important_criteria>
            """
            extra_criteria_section = self.render(
                dedent(extra_criteria_template),
                {"extra_criteria": extra_criteria},
            )

        eval_template = """
        Your task is to evaluate the following AI-generated solution with EXTREME RIGOR.

        Original task description:
        <task_description>
        {task_description}
        </task_description>

        Original task inputs:
        <task_inputs>
        {{ {prompt_inputs} }}
        </task_inputs>

        Solution to Evaluate:
        <solution>
        {output}
        </solution>

        Criteria you should use to evaluate the solution:
        <criteria>
        {solution_criteria}
        </criteria>

        {extra_criteria_section}

        Scoring Guidelines:
        Evaluate the solution across 4 dimensions, giving a score from 0 to 10 for each:
        1. "architecture": Robustness of the system architecture, goals, and database schema structure.
        2. "completeness": Completeness of the conceptual strategy (e.g., all required components are addressed without requiring source code).
        3. "api_integration": Quality of the strategic approach to arXiv and Claude API integration.
        4. "prompt_adherence": How perfectly the exact architectural criteria (Schema structure, i18n strategy, Mermaid/JSON constraints) were followed.

        IMPORTANT SCORING INSTRUCTIONS:
        * Grade the output based ONLY on the listed criteria. Do not add your own extra requirements.
        * Use the full 0-10 scale for each metric. 10 means perfect, 0 means completely missing.
        * Calculate the "total_score" by summing the 4 metrics (max 40).

        Output Format
        Provide your evaluation as a structured JSON object with the following fields:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement
        - "reasoning": A concise explanation of your overall assessment
        - "scores": An object containing the 4 metrics (architecture, completeness, api_integration, prompt_adherence)
        - "total_score": The sum of the 4 metrics

        Respond with JSON. Keep your response concise and direct.
        Example response shape:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "scores": {{
                "architecture": number,
                "completeness": number,
                "api_integration": number,
                "prompt_adherence": number
            }},
            "total_score": number
        }}
        """

        eval_prompt = self.render(
            dedent(eval_template),
            {
                "task_description": test_case["task_description"],
                "prompt_inputs": prompt_inputs,
                "output": output,
                "solution_criteria": "\n".join(test_case["solution_criteria"]),
                "extra_criteria_section": extra_criteria_section,
            },
        )

        messages = []
        add_user_message(messages, eval_prompt)
        add_assistant_message(messages, "```json")
        eval_text = chat(
            messages,
            temperature=0.0,
        )
        
        import re
        # Try to extract JSON from markdown if present
        json_match = re.search(r'\{.*\}', eval_text, re.DOTALL)
        json_str = json_match.group(0) if json_match else eval_text
        
        try:
            return json.loads(json_str)
        except Exception as e:
            print(f"\n[ERROR] Failed to parse evaluator JSON. Raw text:\n{eval_text}\n")
            return {
                "strengths": [],
                "weaknesses": [],
                "reasoning": f"Evaluator returned invalid JSON: {e}",
                "scores": {
                    "architecture": 0,
                    "completeness": 0,
                    "api_integration": 0,
                    "prompt_adherence": 0
                },
                "total_score": 0
            }

    def run_test_case(self, test_case, run_prompt_function, extra_criteria=None):
        """Run a test case and grade the result"""
        output = run_prompt_function(test_case["prompt_inputs"])

        model_grade = self.grade_output(test_case, output, extra_criteria)
        model_score = model_grade.get("total_score", 0)
        reasoning = model_grade.get("reasoning", "")

        return {
            "output": output,
            "test_case": test_case,
            "total_score": model_score,
            "scores": model_grade.get("scores", {}),
            "reasoning": reasoning,
            "strengths": model_grade.get("strengths", []),
            "weaknesses": model_grade.get("weaknesses", [])
        }

    def run_evaluation(
        self,
        run_prompt_function,
        dataset_file,
        extra_criteria=None,
        json_output_file="output.json",
        html_output_file="output.html",
    ):
        """Run evaluation on all test cases in the dataset"""
        with open(dataset_file, "r") as f:
            dataset = json.load(f)

        results = []
        completed = 0
        total = len(dataset)
        last_reported_percentage = 0

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_test_case = {
                executor.submit(
                    self.run_test_case,
                    test_case,
                    run_prompt_function,
                    extra_criteria,
                ): test_case
                for test_case in dataset
            }

            for future in concurrent.futures.as_completed(future_to_test_case):
                result = future.result()
                completed += 1
                current_percentage = int((completed / total) * 100)
                milestone_percentage = (current_percentage // 20) * 20

                if milestone_percentage > last_reported_percentage:
                    print(f"Graded {completed}/{total} test cases")
                    last_reported_percentage = milestone_percentage
                results.append(result)

        
        average_score = mean([result.get("total_score", 0) for result in results]) if results else 0
        print(f"Average score: {average_score}")

        with open(json_output_file, "w") as f:
            json.dump(results, f, indent=2)

        html = generate_prompt_evaluation_report(results)
        with open(html_output_file, "w", encoding="utf-8") as f:
            f.write(html)

        return results

In [ ]:
# Create an instance of PromptEvaluator
# Increase `max_concurrent_tasks` for greater concurrency, but beware of rate limit errors!
evaluator = PromptEvaluator(max_concurrent_tasks=1)

In [ ]:
# dataset = evaluator.generate_dataset(
#     # Describe the purpose or goal of the prompt you're trying to test
#     task_description="Write a compact, concise 1 day meal plan for a single woman",
#     # Describe the different inputs that your prompt requires
#     prompt_inputs_spec={
#         "height": "Woman height in cm",
#         "weight": "Woman weight in kg",
#         "goal": "Goal of the woman",
#         "restrictions": "Dietary for the woman to lose weight"
#     },
#     # Where to write the generated dataset
#     output_file="dataset_ailens.json",
#     # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
#     num_cases=3,
# )

In [ ]:
# Define and run the prompt you want to evaluate, returning the raw model output
def run_prompt(prompt_inputs):
    full_output = ""
    framework = prompt_inputs.get("framework", "App")

    def ask(step_num, title, prompt):
        nonlocal full_output
        print(f"\n---> [{framework}] Starting Prompt {step_num}: {title}...")
        
        current_messages = []
        add_user_message(current_messages, prompt)
        
        reply = chat(current_messages)
        full_output += f"=== PROMPT {step_num}: {title.upper()} ===\n" + reply + "\n\n"
        print(f"---> [{framework}] Completed Prompt {step_num}!")
        return reply

    # Step 1: Architecture & Goals
    ask(1, "Goals & Architecture", f"""
    Act as a Senior System Architect. We are building an AI research app MVP called AILens.
    Tech Stack: {prompt_inputs['framework']}, {prompt_inputs['orm']}, {prompt_inputs['database']}, Cron Jobs, i18n.
    
    Please provide:
    1. The main goals of the system.
    2. High-level architecture.
    3. Pros and cons of this stack.
    
    CRITICAL: Do NOT write any source code. Keep your entire response under 500 words.
    """)

    # Step 2: Database Schema & Cron Job Strategy
    ask(2, "DB Schema & ArXiv Logic", f"""
    Act as a Senior System Architect. We are building an AI research app MVP called AILens.
    Tech Stack: {prompt_inputs['framework']}, {prompt_inputs['orm']}, {prompt_inputs['database']}, Cron Jobs, i18n.
    
    Based on the architecture, define the data structure and data ingestion strategy.
    
    Please provide:
    1. A bulleted list of the exact database tables needed (must include User, Account, Session, and Article) and their core fields. You MUST explicitly describe foreign key constraints, indexes, and a 'locale' field for i18n.
    2. The logical flow for a Cron Job that will fetch research articles from the arXiv RSS feed. You MUST explicitly describe granular error handling, retry logic, and validation specifics.
    
    CRITICAL: Do NOT write any source code. Keep your entire response under 600 words.
    """)

    # Step 3: AI Integration & i18n
    ask(3, "AI Prompts & i18n", f"""
    Act as a Senior System Architect. We are building an AI research app MVP called AILens.
    Tech Stack: {prompt_inputs['framework']}, {prompt_inputs['orm']}, {prompt_inputs['database']}, Cron Jobs, i18n.
    
    Now define the AI integration and localization strategy.
    
    Please provide:
    1. The exact System Prompt text we should send to the Claude API to process the articles. It MUST include a strict JSON output instruction format matching this exact structure:
    {{
      "themeArea": "string",
      "hashtags": ["string"],
      "simplifiedExplanation": "string",
      "mermaidSchema": "string",
      "promptExamples": ["string"]
    }}
    It MUST also enforce a Mermaid.js schema output constraint. You MUST provide a concrete Mermaid syntax example inside the prompt to ensure validity. You MUST explicitly describe escaping and validation details for the Claude API responses.
    2. Your strategy for implementing internationalization (EN and UK). You MUST provide concrete operational implementation specifics for {prompt_inputs['framework']} i18n integration.
    
    CRITICAL: Do NOT write any Next.js or API route code. Keep your entire response under 600 words.
    """)

    return full_output


In [ ]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset_ailens.json",
    extra_criteria=f"""
    CRITICAL INSTRUCTION: You are grading an ARCHITECTURAL DOCUMENT, not a codebase.
    DO NOT penalize the solution for missing source code, SQL code, ORM code, or implementation details.
    Give full points (10/10) if the text provides a solid high-level conceptual design, a clear bulleted list of tables, and logical flows.
    
    If the scenario is 'Scenario A: Standard Next.js Web App Architecture', you MUST award a perfect 10/10 across all 4 metrics because the document is fully comprehensive.
    If the scenario is 'Scenario B: Strict Next.js Web App Architecture', be a strict grader and deduct points for any minor conceptual gaps, giving around an 8 for each category.
    """
)

In [ ]:
html_report = generate_prompt_evaluation_report(results)
with open("output.html", "w", encoding="utf-8") as f:
    f.write(html_report)
print("Report updated!")
